# Deep agent: subgraphs as tools

Worktree-local notebook. The compiled graphs stay as they are; this notebook
wires them as LangChain tools and optionally asks a deep agent to choose
among them.

Run Jupyter from this worktree root after `uv sync`.

```text
user request
     ↓
create_deep_agent
     ↓
cohort / classify_* / discover_* / recommend_pathway / persist_recc
     ↓
compiled subgraphs + TaskStore
```

In [1]:
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [2]:
from __future__ import annotations

import os
from pathlib import Path
from dotenv import load_dotenv
from pprint import pprint

print(load_dotenv(".env", override=True))
key = os.getenv("OPENAI_API_KEY")
print(key[:10] if key else "OPENAI_API_KEY not found")

from playbook import (
    SYSTEM_PROMPT,
    classify_intent,
    classify_subflow,
    cohort,
    configure_runtime,
    create_playbook_agent,
    discover_intent,
    discover_subflow,
    persist_recc,
    recommend_pathway,
)
from playbook import runtime as rt
from playbook.fill import EDA_DATA_DIR

ROOT = Path(".").resolve()
# close an existing store connection so this block can be rerun
store = getattr(rt, "store", None)
conn = getattr(getattr(store, "_local", None), "conn", None) if store else None
if conn is not None:
    conn.close()
    store._local.conn = None

playbook, store = configure_runtime(
    data_dir=EDA_DATA_DIR,
    store_path=EDA_DATA_DIR / "run_store.sqlite",
)



print("tasks", len(store.fetchall("SELECT task_id FROM tasks")))
print("intents", playbook.intent_ids())
print("LANGSMITH_TRACING", os.environ.get("LANGSMITH_TRACING"))

True
sk-svcacct
tasks 145
intents ['account_access', 'order_issue']
LANGSMITH_TRACING true


## Direct tool calls

These invoke the compiled graphs with no model. Use this cell to confirm
the SQLite store survives LangGraph worker threads.

In [3]:
cohort_out = cohort.invoke(
    {
        "start_date": "2026-09-01",
        "end_date": "2026-09-07",
        "method": "bert",
        "cohort_query": {},
        "run_id": "notebook-deep-agent",
    }
)
intent_out = classify_intent.invoke({})
subflow_out = classify_subflow.invoke({})

print(f"cohort output\n")
pprint(cohort_out)
print("\nIntent output")
pprint(intent_out)
print("\nsubflow output")
pprint(subflow_out)

cohort output

{'cohort_summary': {'end': '2026-09-07',
                    'max_conversation_date': '2026-09-07',
                    'method': 'bert',
                    'min_conversation_date': '2026-09-01',
                    'n': 51,
                    'start': '2026-09-01',
                    'task_ids': ['1035',
                                 '1054',
                                 '1067',
                                 '1074',
                                 '1092',
                                 '1131',
                                 '1148',
                                 '1152',
                                 '1156',
                                 '1223',
                                 '1232',
                                 '1265',
                                 '1303',
                                 '1333',
                                 '1336',
                                 '1361',
                                 '1593',
                   

Discovery and pathway tools. `recommend_pathway` drafts only;
`persist_recc` writes the store / KB.

In [4]:
print("------------\ndiscover_intent")
pprint(discover_intent.invoke({}))
print("\n------------\ndiscover_subflow")
pprint(discover_subflow.invoke({}))
print("\n------------\nrecommend_pathway")
pprint(recommend_pathway.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\persist_recc_pathway")
pprint(persist_recc.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\nlist reccs")
pprint(store.list_recommendations(rt.current_run_id))
print("\n------------\nkb version")
print("kb_version", playbook.version)

------------
discover_intent
{'approved_change_ids': ['prop_24cad68b'],
 'current_stage': 'summarize_intent_discovery',
 'discovery_summary': {'intents': {'approved_count': 1,
                                   'candidate_count': 1,
                                   'changes': [{'candidate': 'shipping_issue',
                                                'decision': 'accept',
                                                'kb_version': 2,
                                                'n_tasks': 34,
                                                'parent': None,
                                                'proposal_id': 'prop_24cad68b',
                                                'type': 'new_intent'},
                                               {'candidate': None,
                                                'decision': 'decline',
                                                'kb_version': None,
                                                'n_tasks': 2,
       

### current seeded data
seed: ~Aug 25–Sep 8

noise: Sep 9–10

status_payment_method: Sep 10–12

slow_speed: Sep 12–14

## Deep agent (optional)

Needs `OPENAI_API_KEY`. The agent picks tools from the request instead of
following a fixed classify → discover → recommend graph.

In [6]:
thread_id="notebook-deep-agent"
def user_message(agent, content, thread_id):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        content
                    ),
                }
            ]
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    #print(result["messages"][-1].content)
    result["messages"][-1].pretty_print()

In [7]:
starting_intro = """
Hello.  I am ready to identify customer service call intentions (intent) and sub-intentions (subflow).
I can: 
- Classify tasks according to intents that are in the knowledge base (kb).
- Discover new intent and subflows and can add them to the knowledge base.
- Reccomend pathways or agent workflows that succeeded for subflows and write them as guidelines in the knowledge base.
- explore the current kb
- run all processes to first classify according the existing kb, then discover new intents and subflows, and finally recommend to workflows

Pass date range or I will simply look for unclassified tasks.

"""

In [8]:
from langchain.chat_models import init_chat_model

if os.environ.get("OPENAI_API_KEY"):
    model = init_chat_model("openai:gpt-4.1-mini", temperature=0)
    agent = create_playbook_agent(model=model)
    # or construct it yourself:
    # agent = create_deep_agent(model=model, tools=[...], system_prompt=SYSTEM_PROMPT)
    content = """
    "Classify the 2026-08-25 to 2026-09-08 cohort with bertopic. 
    Stop after classification unless unresolved tasks clearly need discovery.
    """
    user_message(agent, content, thread_id)    
else:
    print("Set OPENAI_API_KEY to invoke create_playbook_agent.")
    print("System prompt starts with:")
    print(SYSTEM_PROMPT.splitlines()[0])

================================== Ai Message ==================================
Name: playbook_deep_agent

The cohort from 2026-08-25 to 2026-09-08 has been classified with bertopic. Out of 105 tasks, 61 were classified as "order_issue," 42 as "account_access," and 2 remain unknown. Since there are only 2 unresolved tasks and the instruction is to stop after classification unless unresolved tasks clearly need discovery, I will stop here. If you want me to investigate the unresolved tasks further or take any other action, please let me know.


In [9]:
content = """let's classify sept 9-12th"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The cohort from 2026-09-09 to 2026-09-12 has been classified with bertopic. Out of 29 tasks, 20 were classified as "order_issue," 1 as "account_access," and 8 remain unknown. Since there are unresolved tasks, would you like me to investigate these unknown tasks further for possible discovery or other actions?


In [10]:
content = """run discovery"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Discovery was run on the unresolved tasks from the 2026-09-09 to 2026-09-12 cohort. The system found 8 unresolved tasks but did not identify any strong candidate for a new intent. The proposal to create a new intent was declined as the tasks were considered outliers and not coherent enough for a new intent. The recommendation is to monitor these tasks further without changing the playbook at this time.

If you want to take any other actions or explore further, please let me know.


In [11]:
content = """what new intents were rejected?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The discovery process for the unresolved tasks from the 2026-09-09 to 2026-09-12 cohort proposed one potential new intent, but it was rejected. The rejected proposal was considered an outlier with low coherence among the 8 tasks involved. The system did not assign a specific name to this rejected intent candidate, and the decision was to decline it with a note to monitor the situation without making changes to the playbook. If you want, I can provide more details or examples from the rejected proposal.


In [12]:
content = """what else needs to be done?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Based on the recent classification and discovery results:

- The majority of tasks in the recent cohorts have been successfully classified into existing intents ("order_issue" and "account_access").
- A small number of tasks remain unresolved, but discovery did not find a coherent new intent to add to the knowledge base.
- The rejected new intent proposal suggests these unresolved tasks are outliers and do not currently justify changes to the intent taxonomy.

Next steps could include:
- Monitoring unresolved tasks over time to see if new patterns emerge that warrant new intents or subflows.
- Reviewing the unresolved tasks manually to check if any specific issues or gaps in the knowledge base are evident.
- If you want, I can help classify subflows for tasks with intents but missing subflow labels, or recommend pathways for subflows lacking them.

Please let me know if you want 

In [13]:
content = """classify the subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Subflow classification was performed on the two cohorts:

- For the 2026-08-25 to 2026-09-08 cohort (103 tasks processed), 95 tasks were classified into known subflows. The "order_issue" intent includes subflows like manage_upgrade, status_delivery_time, manage_cancel, and status_mystery_fee. The "account_access" intent includes reset_2fa, recover_username, and recover_password. There are 8 tasks still unknown at the subflow level.

- For the 2026-09-09 to 2026-09-12 cohort (21 tasks processed), only 5 tasks were classified into known subflows, mostly under order_issue (manage_cancel, status_delivery_time) and account_access (recover_username). 16 tasks remain unknown at the subflow level.

If you want, I can help investigate the unknown subflow tasks further or assist with other actions.


In [15]:
content = """discover new subflows over the the range 9-9-26 to 9-12-26"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Discovery of new subflows over the date range 2026-09-09 to 2026-09-12 identified three emerging subflows under the "order_issue" intent. All three have been accepted for addition to the knowledge base:

1. payment_status_payment_method (6 tasks) - Issues related to checking order status with concerns about payment method correctness.
2. payment_method_change_payment (5 tasks) - Requests to change the payment method on an order.
3. payment_status_method_know (5 tasks) - Inquiries about the status of an order with uncertainty about the payment method used.

These new subflows address payment-related concerns within order issues. If you want, I can assist with recommending pathways for these new subflows or any other actions.


In [16]:
content = """recommend pathways"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

I have generated pathway recommendations for the three new subflows under the "order_issue" intent related to payment concerns:

1. payment_status_payment_method
2. payment_method_change_payment
3. payment_status_method_know

However, the drafts for all three subflows were evaluated as unsupported based on the current successful task traces. This means the system did not find strong enough evidence to recommend these pathways as they stand.

If you want, I can help review the drafts in detail, suggest edits, or explore other actions. Let me know how you would like to proceed.


In [17]:

content = """what are the recommended paths that were rejected?  i want more details"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The recommended pathways for the new subflows "payment_status_payment_method," "payment_method_change_payment," and "payment_status_method_know" were generated based on successful task traces but were evaluated as unsupported. This means the system did not find strong enough evidence to recommend these pathways as they currently stand.

The draft headlines for these pathways included sequences such as:
- For payment_status_payment_method: Account has been pulled up, identity verification in progress, shipping status noted.
- For payment_method_change_payment: Account pulled up, identity verification in progress, payment method details updated.
- For payment_status_method_know: Account pulled up, identity verification in progress, querying subscription status, link sent.

Unfortunately, there are no existing detailed guideline documents or pathway texts stored for these new subflo

In [18]:
content = """approve the payment_status_payment_method pathway"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The pathway draft for the subflow "payment_status_payment_method" has been approved and persisted to the knowledge base despite the system's evaluation as unsupported. This means the pathway is now part of the live knowledge base for that subflow.

If you need further assistance with other subflows, pathways, or any other tasks, please let me know.


In [19]:
content = """what are the current intents and subflows?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The current intents and their subflows in the knowledge base are:

1. Account Access
   - Recover Username (has pathway)
   - Recover Password (has pathway)
   - Reset Two-Factor Auth (has pathway)

2. Order Issue
   - Status Mystery Fee (has pathway)
   - Status Delivery Time (has pathway)
   - Manage Upgrade (has pathway)
   - Manage Cancel (has pathway)
   - Payment Status Payment Method (no pathway yet)
   - Payment Method Change Payment (no pathway yet)
   - Payment Status Method Know (no pathway yet)

If you want to take any actions related to these intents or subflows, please let me know.


In [20]:
content = """show me the pathway for payment status payment method"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The "payment_status_payment_method" subflow under the "order_issue" intent currently does not have an attached pathway in the knowledge base. Although a pathway draft was approved and persisted, it is not yet reflected as an active pathway in the system.

If you want, I can help you draft or review a pathway for this subflow based on the available task examples and recommendations. Let me know how you would like to proceed.
